# Тестовое задание: Анализ оттока и эффективности маркетинговой кампании

В базе данных хранится 2 таблицы: pl и campaign. Ключами для связи этих таблиц является поле client_id. В таблице pl находится информация о ЧОД (чистый операционный доход) от клиента с продуктом в разрезе периодов. В таблице campaign хранятся данные о прошедшей между отчетными периодами "p_1" и "p_2" маркетинговой кампании.

**Атрибутный состав таблицы pl:**
- client_id - id клиента
- segment - сегмент клиента внутри СМБ(Средний и малый бизнес)
- period - период расчета ЧОД банка от продукта, используемого клиентом
- product - продукт
- pl - ЧОД в тысячах рублей </br>

Пример: строка 0 на срезе выше - в периоде "p_1" за продукт "Кредит" с клиента "yxwtZv530y" массового сегмента банк заработал 0.374924 тыс. рублей.

Если в каком-либо периоде информация о клиенте/продукте у клиента отсутствует, значит в этот период клиента в банке не было/клиент продуктом не пользовался.

**Атрибутный состав таблицы campaign:**
- client_id - id клиента
- product - продукт кампании
- response - статус отклика на предложение
- manager_id - id менеджера, на которого назначен клиент

## Задание
1. Посчитать средний ЧОД по продуктам и количество клиентов с продуктом для каждого сегмента в разрезе периодов.
2. Посчитать средний ЧОД и количество клиентов в каждом сегменте в разрезе периодов.
3. Оценить отток клиентов с продуктов в периоде p_2.
4. Посчитать сумму ЧОД, привлеченную каждым менеджером в ходе маркетинговой кампании
5. Оценить эффективность маркетинговой кампании.

### Подготовка данных и исследовательский анализ (EDA)

In [1]:
import os
import sqlite3 as sql
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format

if not os.path.exists('database.db'):
    raise FileNotFoundError('Файл database.db не найден: положите его рядом с ноутбуком')

db = sql.connect('database.db')
df_pl = pd.read_sql_query('select * from pl', db)
df_campaign = pd.read_sql_query('select * from campaign', db)

In [2]:
df_pl.head(2)

,client_id,segment,period,product,pl
0,ZDosW1zUAi,Массовый,p_1,Кредит,9.02
1,wVRHQ9hzrw,Массовый,p_1,Кредит,18.89


In [3]:
df_campaign.head(2)

,client_id,product,response,manager_id
0,owOyJEIst9,Кредит,Согласие,CM_EiBeeRMhdA
1,HkhuPN1KG7,Кредит,Не обработано менеджером,CM_PzEEav4ic0


In [4]:
df_pl.isna().sum()

client_id    0
segment      0
period       0
product      0
pl           0
dtype: int64

In [5]:
df_campaign.isna().sum()

client_id     0
product       0
response      0
manager_id    0
dtype: int64

### Задание 1. Средний ЧОД и количество клиентов по продуктам

In [6]:
task1 = df_pl.groupby(['segment', 'period', 'product']).agg({'pl' : 'mean', 'client_id' : 'nunique'})
task1

pl  client_id
segment     period product                           
Верхний     p_1    Банковская карта  60.57         49
                   Депозит          112.47        254
                   Кредит           236.94        360
                   РКО               22.85       1149
                   Эквайринг         80.52        183
            p_2    Банковская карта  51.77        167
                   Депозит           99.26        283
                   Кредит           248.19        548
                   РКО               22.06       1162
                   Эквайринг         72.97        284
Массовый    p_1    Банковская карта  13.31        895
                   Депозит           28.75       3537
                   Кредит            59.22       5247
                   РКО                5.80      16728
                   Эквайринг         19.63       2625
            p_2    Банковская карта  11.73       2636
                   Депозит           25.22       4345
                   Кредит            67.11       7845
                   РКО                5.57      16700
                   Эквайринг         17.35       4424
Стандартный p_1    Банковская карта  26.42        338
                   Депозит           60.99       1292
                   Кредит           120.10       1871
                   РКО               11.42       5907
                   Эквайринг         39.33        962
            p_2    Банковская карта  22.44        917
                   Депозит           49.33       1570
                   Кредит           131.61       2871
                   РКО               10.98       5870
                   Эквайринг         35.05       1590

### Задание 2. Средний ЧОД и количество клиентов в сегментах

In [7]:
task2 = df_pl.groupby(['segment', 'period']).agg({'pl' : 'mean', 'client_id' : 'nunique'})
task2

pl  client_id
segment     period                 
Верхний     p_1    79.11       1195
            p_2    89.65       1206
Массовый    p_1    19.73      17207
            p_2    23.27      17330
Стандартный p_1    40.28       6061
            p_2    46.50       6110

### Задание 3. Оценка оттока клиентов в периоде p_2

In [8]:
df_p1, df_p2 = df_pl[df_pl['period'] == 'p_1'], df_pl[df_pl['period'] == 'p_2']

no_clients = df_p1[~df_p1['client_id'].isin(df_p2['client_id'])]
task3 = no_clients['client_id'].nunique()

pairs_p1, pairs_p2 = set(zip(df_p1['client_id'], df_p1['product'])), set(zip(df_p2['client_id'], df_p2['product']))
lost_pairs = pairs_p1 - pairs_p2
task3_products = len(lost_pairs)

print(f'Отток клиентов (ушли из банка): {task3} человек')
print(f'Отток с продуктов (клиент + продукт): {task3_products} человек')
print(pd.Series([product for client, product in lost_pairs]).value_counts())

Отток клиентов (ушли из банка): 346 человек
Отток с продуктов (клиент + продукт): 12979 человек
Кредит              4084
Депозит             3824
Эквайринг           2785
РКО                 1207
Банковская карта    1079
Name: count, dtype: int64


### Задание 4. Сумма ЧОД по менеджерам

In [9]:
df_campaign.head(0)

,client_id,product,response,manager_id


In [10]:
df_campaign['response'].unique()

array(['Согласие', 'Не обработано менеджером', 'Недозвон', 'Перезвонить',
       'Отказ'], dtype=object)

In [11]:
df_merged = pd.merge(df_campaign, df_p2, on = ['client_id', 'product'])
success_tothe_manager =  df_merged[df_merged['response'] == 'Согласие']
task4 = success_tothe_manager.groupby('manager_id').agg({'pl' : 'sum'})
task4

,pl
manager_id,
CM_2755B4KTu4,"16,600.10"
CM_6ofody41cY,"13,407.51"
CM_70HjN0EVqf,"11,831.28"
CM_9vMKq9aqw9,"11,080.97"
CM_EiBeeRMhdA,"12,820.88"
CM_PzEEav4ic0,"12,933.25"
CM_R7BdTh0Svt,"14,119.01"
CM_RCwv1r0OHO,"12,056.91"
CM_c28lGRRaEI,"13,358.89"


### Задание 5. Оценка эффективности маркетинговой кампании

In [12]:
clients_response = df_campaign['response'].value_counts(normalize = True) * 100
print(f'доля ответов:\n{clients_response}\n')

revenue_campaign = task4['pl'].sum()

agreed = df_campaign[df_campaign['response'] == 'Согласие']
agreed_pairs = set(zip(agreed['client_id'], agreed['product']))
p1_pairs = set(zip(df_p1['client_id'], df_p1['product']))

print(f'ЧОД в p_2 по продуктам кампании у согласившихся: {revenue_campaign:,.0f} тыс. рублей')
print(f'Пар "клиент + продукт" у согласившихся, которые уже были в p_1: {len(agreed_pairs & p1_pairs)} из {len(agreed_pairs)}')
db.close()

доля ответов:
response
Согласие                   56.32
Недозвон                   14.98
Отказ                      14.34
Не обработано менеджером    9.68
Перезвонить                 4.68
Name: proportion, dtype: float64

ЧОД в p_2 по продуктам кампании у согласившихся: 256,044 тыс. рублей
Пар "клиент + продукт" у согласившихся, которые уже были в p_1: 0 из 2816


#### Выводы по результатам маркетинговой кампании:
- **Финансовый результат:** среди согласившихся клиентов ЧОД по продуктам кампании в p_2 составил около 256 млн рублей. В p_1 по этим парам "клиент + продукт" ЧОД равен нулю (все 2816 пар новые), поэтому весь доход относится к кампании. Но контрольной группы нет, поэтому часть клиентов могла бы открыть продукт и без неё.

- **Воронка продаж:** согласие дали 56.3% клиентов, отказ - 14.3%, недозвон - 15.0%, "перезвонить" - 4.7%.

- **Зоны роста (упущенная выгода):** Почти 10% выделенной базы имеют статус "Не обработано менеджером". Организация дополнительного контроля за прозвоном этой группы клиентов позволит довести часть из них до "Согласия" и принести банку дополнительную выручку без дополнительных затрат на новую кампанию.